# E-Commerce Data Cleaning, Manipulation & Feature Engineering
**Author:** Piyush ([@piyush8769](https://github.com/piyush8769))

This notebook provides a complete, step-by-step data analysis workflow.
Each code block is kept **simple and focused on a single task**, with clear comments explaining:
1. **The Problem** (what is wrong in the raw data)
2. **How to Find It** (diagnostic check)
3. **How to Resolve It** (clean, simple pandas method)


### Block 0: Load Dataset and Inspect Shape
We load the raw CSV file into a pandas DataFrame and check the initial row and column counts.


In [1]:
# Load raw dataset
import pandas as pd
import numpy as np

df = pd.read_csv("ecommerce_customer_data.csv")

# Check rows and columns
print(f"Loaded dataset successfully.")
print(f"Initial Shape: {df.shape[0]} rows, {df.shape[1]} columns")


Loaded dataset successfully.
Initial Shape: 35350 rows, 22 columns


### Block 1: Find and Remove Duplicate Rows
**Problem:** In production databases, network timeouts or retried batch jobs often cause duplicate transactions.  
**Resolution:** Count duplicates using `df.duplicated().sum()`, then remove them with `df.drop_duplicates()`.


In [2]:
# Check how many duplicate rows exist
duplicate_count = df.duplicated().sum()
print(f"Found {duplicate_count} exact duplicate rows.")

# Drop duplicate rows and keep the first occurrence
df = df.drop_duplicates()
print(f"Remaining rows after deduplication: {len(df)}")


Found 350 exact duplicate rows.
Remaining rows after deduplication: 35000


### Block 2: Standardize Product Category Names
**Problem:** Product categories have inconsistent casing, leading spaces, and typos (e.g. `'Eletronics'`, `'electr.'`).  
**Resolution:** Strip whitespace, convert to Title Case, and map typos using a simple replacement dictionary.


In [3]:
# Strip spaces and convert to Title Case
df['product_category'] = df['product_category'].astype(str).str.strip().str.title()

# Dictionary to fix typos
category_mapping = {
    'Eletronics': 'Electronics',
    'Electr.': 'Electronics',
    'Apparel & Clothing': 'Clothing',
    'Home And Kitchen': 'Home & Kitchen',
    'Home': 'Home & Kitchen',
    'Beauty & Care': 'Beauty & Personal Care',
    'Beauty': 'Beauty & Personal Care',
    'Sports': 'Sports & Outdoors'
}

# Replace typos with standard names
df['product_category'] = df['product_category'].replace(category_mapping)

# Verify clean categories
print("Cleaned Product Categories:")
print(df['product_category'].value_counts())


Cleaned Product Categories:
product_category
Sports & Outdoors         7186
Home & Kitchen            6978
Electronics               6976
Clothing                  6964
Beauty & Personal Care    6896
Name: count, dtype: int64


### Block 3: Standardize Country Names
**Problem:** Country names are mixed between abbreviations (`'US'`, `'IND'`, `'DE'`) and full names (`'United States'`).  
**Resolution:** Map abbreviations to clean unified country names.


In [4]:
# Strip any whitespace
df['customer_country'] = df['customer_country'].astype(str).str.strip()

# Mapping dictionary for country variations
country_mapping = {
    'US': 'USA',
    'United States': 'USA',
    'U.S.A.': 'USA',
    'IND': 'India',
    'DE': 'Germany',
    'UK': 'United Kingdom',
    'CAN': 'Canada',
    'AUS': 'Australia'
}

# Replace variations
df['customer_country'] = df['customer_country'].replace(country_mapping)

# Check results
print("Cleaned Country Counts:")
print(df['customer_country'].value_counts())


Cleaned Country Counts:
customer_country
India             6134
Australia         6018
Canada            5817
USA               5737
United Kingdom    5665
Germany           5629
Name: count, dtype: int64


### Block 4: Standardize Demographics (Gender & Membership Tier)
**Problem:** Gender contains `'Femal'`, `'Mle'`, and empty strings. Membership tier has mixed casing.  
**Resolution:** Clean gender using dictionary mapping and capitalize membership tiers.


In [5]:
# Clean gender typos and placeholders
gender_mapping = {
    'M': 'Male',
    'Mle': 'Male',
    'F': 'Female',
    'Femal': 'Female',
    '': np.nan,
    'Unknown': np.nan,
    'None': np.nan
}
df['customer_gender'] = df['customer_gender'].replace(gender_mapping)

# Clean Membership Tier: standardize casing and map VIP to Platinum
df['membership_tier'] = df['membership_tier'].astype(str).str.strip().str.capitalize()
df['membership_tier'] = df['membership_tier'].replace({'None': np.nan, 'Nan': np.nan, 'Vip': 'Platinum'})

print("Cleaned Membership Tiers:")
print(df['membership_tier'].value_counts(dropna=False))


Cleaned Membership Tiers:
membership_tier
Silver      9060
Platinum    9015
Bronze      8605
Gold        4249
NaN         4071
Name: count, dtype: int64


### Block 5: Clean Currency Columns (`unit_price` & `total_amount`)
**Problem:** Prices are stored as strings containing symbols (`$`, `€`, `USD`), commas, and text like `'missing'`.  
**Resolution:** Remove symbols, convert to numeric with `pd.to_numeric()`, and fix negative values using `.abs()`.


In [6]:
# Clean unit_price: remove symbols and commas
df['unit_price'] = df['unit_price'].astype(str)
df['unit_price'] = df['unit_price'].str.replace('$', '', regex=False)
df['unit_price'] = df['unit_price'].str.replace('€', '', regex=False)
df['unit_price'] = df['unit_price'].str.replace('USD', '', regex=False)
df['unit_price'] = df['unit_price'].str.replace(',', '', regex=False)
df['unit_price'] = df['unit_price'].str.strip()

# Convert to float (invalid words like 'missing' become NaN)
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

# Fix negative price errors to positive
df['unit_price'] = df['unit_price'].abs()

# Clean total_amount in the exact same simple way
df['total_amount'] = df['total_amount'].astype(str)
df['total_amount'] = df['total_amount'].str.replace('$', '', regex=False)
df['total_amount'] = df['total_amount'].str.replace('€', '', regex=False)
df['total_amount'] = df['total_amount'].str.replace('USD', '', regex=False)
df['total_amount'] = df['total_amount'].str.replace(',', '', regex=False)
df['total_amount'] = df['total_amount'].str.replace('N/A', '', regex=False)
df['total_amount'] = df['total_amount'].str.strip()
df['total_amount'] = pd.to_numeric(df['total_amount'], errors='coerce')

print(f"Average unit price: ${df['unit_price'].mean():.2f}")
print(f"Average total amount: ${df['total_amount'].mean():.2f}")


Average unit price: $152.35
Average total amount: $8314.15


### Block 6: Clean Discount Percentages
**Problem:** Discounts are stored in mixed styles: `'15%'`, `0.15`, `'?'`, and `None`.  
**Resolution:** Convert all values to a standardized float ratio between `0.0` and `1.0`.


In [7]:
def clean_single_discount(val):
    # Check for empty or invalid placeholder values
    if pd.isna(val) or str(val).strip() in ['?', 'None', 'nan', '']:
        return 0.0
    text = str(val).replace('%', '').strip()
    try:
        num = float(text)
        # If entered as whole percentage like 15, divide by 100 to get 0.15
        if num > 1.0:
            return num / 100.0
        return num
    except ValueError:
        return 0.0

# Apply cleaning function to discount column
df['discount_applied'] = df['discount_applied'].apply(clean_single_discount)
print(f"Discount range: Min = {df['discount_applied'].min()}, Max = {df['discount_applied'].max()}")


Discount range: Min = 0.0, Max = 0.5


### Block 7: Parse Mixed Date Formats
**Problem:** `order_date` contains multiple date styles (`YYYY-MM-DD`, `DD/MM/YYYY`, `Month Day, Year`) and corrupted text.  
**Resolution:** Parse with `pd.to_datetime(format='mixed', errors='coerce')` and drop rows where dates are invalid.


In [8]:
# Convert order date with mixed format support
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')
df['customer_signup_date'] = pd.to_datetime(df['customer_signup_date'], errors='coerce')

# Drop rows with unparseable order dates
before_drop = len(df)
df = df.dropna(subset=['order_date'])
print(f"Dropped {before_drop - len(df)} rows with corrupted order dates.")
print(f"Valid transactions remaining: {len(df)}")


Dropped 704 rows with corrupted order dates.
Valid transactions remaining: 34296


### Block 8: Treat Outliers and Logical Errors
**Problem:** Impossible ages ($-5$, $150$), negative quantities, and missing totals.  
**Resolution:** Cap invalid ages and quantities to `NaN`, fill missing quantities with $1$, and recalculate missing totals using:
$$\text{total} = \text{unit\_price} \times \text{quantity} \times (1 - \text{discount})$$


In [9]:
# Age: set impossible values (< 18 or > 100) to NaN
df.loc[(df['customer_age'] < 18) | (df['customer_age'] > 100), 'customer_age'] = np.nan

# Quantity: set negative or extreme values (> 100) to NaN
df.loc[(df['quantity'] <= 0) | (df['quantity'] > 100), 'quantity'] = np.nan

# Fill missing quantity with default 1.0
df['quantity'] = df['quantity'].fillna(1.0)

# Set negative delivery days to NaN
df.loc[df['delivery_time_days'] < 0, 'delivery_time_days'] = np.nan

# Recompute missing total_amount where missing
expected_total = df['unit_price'] * df['quantity'] * (1 - df['discount_applied'])
df['total_amount'] = df['total_amount'].fillna(expected_total.round(2))

print(f"Data cleaning complete. Clean dataset row count: {len(df)}")


Data cleaning complete. Clean dataset row count: 34296


### Block 9: Customer-Level Aggregation
**Business Purpose:** Roll up order transactions to the customer level to see lifetime spend, order count, and lifespan.


In [10]:
# Group by customer_id
customer_summary = df.groupby('customer_id').agg(
    total_spend=('total_amount', 'sum'),
    total_orders=('order_id', 'nunique'),
    avg_order_value=('total_amount', 'mean'),
    first_order=('order_date', 'min'),
    last_order=('order_date', 'max')
).reset_index()

# Calculate customer lifespan in days
customer_summary['lifespan_days'] = (
    customer_summary['last_order'] - customer_summary['first_order']
).dt.days

print(f"Calculated customer metrics for {len(customer_summary)} unique customers:")
print(customer_summary.head(5))


Calculated customer metrics for 5984 unique customers:
  customer_id  total_spend  total_orders  ...  first_order last_order lifespan_days
0  CUST-00001      1268.98             9  ...   2023-01-03 2024-05-20           503
1  CUST-00002      2754.05             6  ...   2020-06-16 2022-02-14           608
2  CUST-00003    209661.28             6  ...   2021-03-29 2022-11-25           606
3  CUST-00004      1465.90             9  ...   2021-05-22 2023-05-13           721
4  CUST-00005      2286.92             4  ...   2020-10-26 2022-09-10           684

[5 rows x 7 columns]


### Block 10: Audit Check (Billing Discrepancies)
**Business Purpose:** Detect transactions where the recorded `total_amount` differs from the calculated formula.


In [11]:
# Calculate expected total
df['expected_total'] = (df['unit_price'] * df['quantity'] * (1 - df['discount_applied'])).round(2)

# Calculate difference
df['price_difference'] = (df['total_amount'] - df['expected_total']).round(2)

# Count discrepancies greater than $1.00
discrepancies = df[df['price_difference'].abs() > 1.0]
print(f"Identified {len(discrepancies)} orders with price discrepancies.")


Identified 2975 orders with price discrepancies.


### Block 11: Feature Engineering - Date & Time Features
**Purpose:** Derive date parts and weekend flags to detect purchasing habits.


In [12]:
# Extract year, month, and day
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day
df['day_of_week'] = df['order_date'].dt.day_name()

# Create is_weekend flag (1 = Saturday/Sunday, 0 = Weekday)
df['is_weekend'] = df['order_date'].dt.dayofweek.isin([5, 6]).astype(int)

# Days as customer at the time of placing this order
df['days_as_customer'] = (df['order_date'] - df['customer_signup_date']).dt.days
df['days_as_customer'] = df['days_as_customer'].clip(lower=0)

print("Engineered date features preview:")
print(df[['order_date', 'day_of_week', 'is_weekend', 'days_as_customer']].head(4))


Engineered date features preview:
  order_date day_of_week  is_weekend  days_as_customer
0 2025-08-29      Friday           0             408.0
1 2023-10-18   Wednesday           0              74.0
2 2024-12-24     Tuesday           0             463.0
4 2023-06-14   Wednesday           0             720.0


### Block 12: Feature Engineering - RFM Metrics
**Purpose:** Calculate **Recency** (days since latest purchase), **Frequency** (number of orders), and **Monetary** (total spent) for marketing segmentation.


In [13]:
# Reference date is 1 day after the latest order in the dataset
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

# Group by customer and compute RFM
rfm = df.groupby('customer_id').agg(
    recency_days=('order_date', lambda x: (snapshot_date - x.max()).days),
    frequency_orders=('order_id', 'nunique'),
    monetary_spend=('total_amount', 'sum')
).reset_index()

# Merge RFM back into the main DataFrame
df = df.merge(rfm, on='customer_id', how='left')

print("RFM metrics merged into main dataset:")
print(df[['customer_id', 'recency_days', 'frequency_orders', 'monetary_spend']].head(4))


RFM metrics merged into main dataset:
  customer_id  recency_days  frequency_orders  monetary_spend
0  CUST-04173           233                14         4437.17
1  CUST-03340           560                 7         3366.88
2  CUST-02349           769                 5          978.25
3  CUST-05455          1329                 6         2164.25


### Block 13: Feature Engineering - Email Domain Extraction
**Purpose:** Extract domain from email to identify whether an order came from a corporate buyer or public webmail.


In [14]:
def extract_domain(email):
    if pd.isna(email) or '@' not in str(email):
        return 'unknown'
    return str(email).split('@')[-1].strip().lower()

df['email_domain'] = df['customer_email'].apply(extract_domain)

# Flag corporate buyers (non-public email services)
public_domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'hotmail.com', 'icloud.com', 'unknown']
df['is_corporate_buyer'] = (~df['email_domain'].isin(public_domains)).astype(int)

print("Top 5 email domains in the dataset:")
print(df['email_domain'].value_counts().head(5))
print(f"Percentage of corporate buyers: {df['is_corporate_buyer'].mean() * 100:.2f}%")


Top 5 email domains in the dataset:
email_domain
outlook.com    5230
icloud.com     5187
hotmail.com    5172
gmail.com      5142
company.org    5075
Name: count, dtype: int64
Percentage of corporate buyers: 20.31%


### Block 14: Feature Engineering - Binning and Encodings
**Purpose:**
1. Discretize continuous age into standard demographic age groups using `pd.cut()`.
2. Ordinally encode `membership_tier` into numeric ranks.
3. One-hot encode `payment_method` using `pd.get_dummies()`.


In [15]:
# 1. Age binning
age_bins = [0, 25, 35, 50, 100]
age_labels = ['18-25', '26-35', '36-50', '50+']
df['age_group'] = pd.cut(df['customer_age'], bins=age_bins, labels=age_labels, right=True)

# 2. Ordinal encoding for Membership Tier
tier_ranking = {'Bronze': 1, 'Silver': 2, 'Gold': 3, 'Platinum': 4}
df['membership_tier_rank'] = df['membership_tier'].map(tier_ranking).fillna(0).astype(int)

# 3. One-hot encoding for payment method
payment_dummies = pd.get_dummies(df['payment_method'], prefix='pay', drop_first=True, dtype=int)
df = pd.concat([df, payment_dummies], axis=1)

print("Encoded age groups:")
print(df['age_group'].value_counts())


Encoded age groups:
age_group
50+      13937
36-50     8253
26-35     5491
18-25     4526
Name: count, dtype: int64


### Block 15: Export Cleaned & Engineered Dataset
Save the final DataFrame to CSV.


In [16]:
output_filename = "ecommerce_cleaned_and_engineered.csv"
df.to_csv(output_filename, index=False)

print("=" * 70)
print(f"Pipeline Completed Successfully!")
print(f"Final Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Exported to file: {output_filename}")
print("=" * 70)


Pipeline Completed Successfully!
Final Dataset Shape: 34296 rows, 45 columns
Exported to file: ecommerce_cleaned_and_engineered.csv
